# בנייה עם דגמי Mistral 

## מבוא 

שיעור זה יכסה: 
- חקירת הדגמים השונים של Mistral 
- הבנת תרחישים ושימושי מקרה עבור כל דגם 
- דוגמאות קוד המראות את התכונות הייחודיות של כל דגם. 


## דגמי Mistral  

בשיעור זה, נחקור 3 דגמי Mistral שונים:  
**Mistral Large**, **Mistral Small** ו-**Mistral Nemo**.  

כל אחד מהדגמים הללו זמין בחינם ב-[Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst). הקוד במחברת זו ישתמש בדגמים אלו כדי להריץ את הקוד.  

> **הערה:** GitHub Models ייסגר בסוף יולי 2026. להלן פרטים נוספים על השימוש ב-[Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) כדי לבצע אב-טיפוס עם דגמי AI.  


## מיסטרל לارج 2 (2407)
מיסטרל לارج 2 הוא כיום הדגם המוביל של מיסטרל ומתוכנן לשימוש ארגוני.

הדגם הוא שדרוג ל־מיסטרל לارج המקורי ומציע
- חלון הקשר גדול יותר - 128k לעומת 32k
- ביצועים טובים יותר במשימות מתמטיקה וקידוד - דיוק ממוצע של 76.9% לעומת 60.4%
- שיפור בביצועים רב־לשוניים - השפות כוללות: אנגלית, צרפתית, גרמנית, ספרדית, איטלקית, פורטוגזית, הולנדית, רוסית, סינית, יפנית, קוריאנית, ערבית והינדי.

עם תכונות אלו, מיסטרל לارج מצטיין ב-
- *הפקה משופרת באמצעות אחזור (RAG)* - בזכות חלון ההקשר הגדול יותר
- *קריאת פונקציות* - לדגם זה יש קריאת פונקציה מובנית שמאפשרת אינטגרציה עם כלים ו-API חיצוניים. קריאות אלו יכולות להיעשות במקביל או אחת אחרי השנייה בסדר רציף.
- *הפקת קוד* - הדגם מצטיין בהפקת קוד בפייתון, ג'אווה, TypeScript ו־C++.


### דוגמת RAG בשימוש במיסטרל לרג' 2 


בדוגמה זו, אנו משתמשים ב-Mistral Large 2 להריץ דפוס RAG על מסמך טקסט. השאלה כתובה בקוריאנית ושואלת על פעילויות המחבר לפני הקולג'.

הוא משתמש במודל ההטבעות של Cohere ליצירת הטבעות של מסמך הטקסט כמו גם של השאלה. בדוגמה זו, הוא משתמש בחבילת Python של faiss כאכסון וקטורי.

הטקסט שנשלח למודל Mistral כולל גם את השאלות וגם את הקטעים שנשלפו והם דומים לשאלה. המודל לאחר מכן נותן תגובה בשפה טבעית.


In [50]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
import numpy as np
import faiss
import os

from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential
from azure.ai.inference import EmbeddingsClient

# Get these from your Microsoft Foundry project's "Overview" page
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Mistral-large"
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = requests.get('https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt')
text = response.text

chunk_size = 2048
chunks = [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]
len(chunks)

embed_model_name = "cohere-embed-v3-multilingual" 

embed_client = EmbeddingsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(token)
)

embed_response = embed_client.embed(
    input=chunks,
    model=embed_model_name
)



text_embeddings = []
for item in embed_response.data:
    length = len(item.embedding)
    text_embeddings.append(item.embedding)
text_embeddings = np.array(text_embeddings)


d = text_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(text_embeddings)

question = "저자가 대학에 오기 전에 주로 했던 두 가지 일은 무엇이었나요?？"

question_embedding = embed_client.embed(
    input=[question],
    model=embed_model_name
)

question_embeddings = np.array(question_embedding.data[0].embedding)


D, I = index.search(question_embeddings.reshape(1, -1), k=2) # distance, index
retrieved_chunks = [chunks[i] for i in I.tolist()[0]]

prompt = f"""
Context information is below.
---------------------
{retrieved_chunks}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {question}
Answer:
"""


chat_response = client.complete(
    messages=[
        SystemMessage(content="You are a helpful assistant."),
        UserMessage(content=prompt),
    ],
    temperature=1.0,
    top_p=1.0,
    max_tokens=1000,
    model=model_name
)

print(chat_response.choices[0].message.content)


The author primarily engaged in two activities before college: writing and programming. In terms of writing, they wrote short stories, albeit not very good ones, with minimal plot and characters expressing strong feelings. For programming, they started writing programs on the IBM 1401 used for data processing during their 9th grade, at the age of 13 or 14. They used an early version of Fortran and typed programs on punch cards, later loading them into the card reader to run the program.


## Mistral Small 
Mistral Small הוא דגם נוסף במשפחת הדגמים Mistral תחת קטגוריית הפרמייר/אנטרפרייז. כפי שהשם מרמז, דגם זה הוא מודל שפה קטן (SLM). היתרונות בשימוש ב-Mistral Small הם שהוא: 
- חוסך עלויות בהשוואה ל-LLMs של Mistral כמו Mistral Large ו-NeMo - הנחה של 80% במחיר
- זמן השהייה נמוך - תגובה מהירה יותר בהשוואה ל-LLMs של Mistral
- גמיש - ניתן לפרוס בסביבות שונות עם פחות הגבלות על המשאבים הנדרשים. 


Mistral Small מעולה ל: 
- משימות מבוססות טקסט כמו סיכום, ניתוח סנטימנט ותרגום. 
- יישומים בהם נעשים בקשות לעיתים קרובות בשל יעילות העלות שלו 
- משימות קוד עם זמן השהייה נמוך כגון סקירה והצעות קוד 


## השוואה בין Mistral Small ל-Mistral Large 

להצגת ההבדלים בזמן תגובה בין Mistral Small ל-Mistral Large, הפעל את התאים למטה. 

עליך לראות הבדל בזמן תגובה בין 3-5 שניות. שים לב גם לאורך התגובות והסגנון עבור אותו פרומפט.  


In [ ]:
import os 
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Mistral-small"
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(content="You are a helpful coding assistant."),
        UserMessage(content="Can you write a Python function to the fizz buzz test?"),
    ],
    temperature=1.0,
    top_p=1.0,
    max_tokens=1000,
    model=model_name
)

print(response.choices[0].message.content)


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Mistral-large"
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(content="You are a helpful coding assistant."),
        UserMessage(content="Can you write a Python function to the fizz buzz test?"),
    ],
    temperature=1.0,
    top_p=1.0,
    max_tokens=1000,
    model=model_name
)

print(response.choices[0].message.content)


## Mistral NeMo

בהשוואה לשני הדגמים האחרים שנדונו בשיעור זה, Mistral NeMo הוא הדגם החופשי היחיד עם רישיון Apache2.

הוא נחשב לשדרוג לגרסת ה-LLM בקוד פתוח הקודמת מ-Mistral, Mistral 7B.

כמה תכונות נוספות של דגם NeMo הן:

- *טוקניזציה יעילה יותר:* דגם זה משתמש בטוקנייזר Tekken במקום בטוקנייזר הנפוץ יותר tiktoken. זה מאפשר ביצועים טובים יותר על שפות וקוד שונים.

- *כיוונון מדויק:* הדגם הבסיסי זמין לכיוונון. זה מאפשר גמישות רבה יותר במקרים בהם נדרש כיוונון.

- *קריאה פונקציונלית מקורית* - בדומה ל-Mistral Large, דגם זה אומן לקריאת פונקציות. זה עושה אותו ייחודי כאחד מהדגמים הראשונים בקוד פתוח שעושים זאת.


## מיסטרל NeMo

בהשוואה לשני המודלים האחרים שדן בהם בשיעור זה, מיסטרל NeMo הוא המודל החינמי היחיד עם רישיון Apache2.

הוא נחשב לשדרוג ל-LLM הקודמת בקוד פתוח של מיסטרל, Mistral 7B.

כמה תכונות נוספות של מודל NeMo הן:

- *טוקניזציה יעילה יותר:* מודל זה משתמש בטוקנייזר Tekken במקום ב-tiktoken שנפוץ יותר. זה מאפשר ביצועים טובים יותר בשפות ובקוד שונים.

- *כיוונון עדין:* המודל הבסיסי זמין לכיוונון עדין. זה מאפשר גמישות רבה יותר במקרים שבהם נדרש כיוונון.

- *קריאה לפונקציות בשפה טבעית* - בדומה למיסטרל Large, מודל זה עבר אימון על קריאה לפונקציות. זה הופך אותו לייחודי כאחד מהמודלים הראשונים בקוד פתוח שעושים זאת.


### השוואת מטפלי טוקנים 

בדוגמה זו, נבחן כיצד Mistral NeMo מטפל בטוקניזציה בהשוואה ל-Mistral Large. 

שתי הדוגמאות משתמשות באותו הפקודה, אך תראו ש-NeMo מחזיר פחות טוקנים לעומת Mistral Large. 


In [11]:
pip install mistral-common

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.0/797.0 kB 15.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [12]:
# Import needed packages:
from mistral_common.protocol.instruct.messages import (
    UserMessage,
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import (
    Function,
    Tool,
)
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

# Load Mistral tokenizer

model_name = "open-mistral-nemo	"

tokenizer = MistralTokenizer.from_model(model_name)

# Tokenize a list of messages
tokenized = tokenizer.encode_chat_completion(
    ChatCompletionRequest(
        tools=[
            Tool(
                function=Function(
                    name="get_current_weather",
                    description="Get the current weather",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "description": "The temperature unit to use. Infer this from the users location.",
                            },
                        },
                        "required": ["location", "format"],
                    },
                )
            )
        ],
        messages=[
            UserMessage(content="What's the weather like today in Paris"),
        ],
        model=model_name,
    )
)
tokens, text = tokenized.tokens, tokenized.text

# Count the number of tokens
print(len(tokens))

128


In [13]:
# Import needed packages:
from mistral_common.protocol.instruct.messages import (
    UserMessage,
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.protocol.instruct.tool_calls import (
    Function,
    Tool,
)
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

# Load Mistral tokenizer

model_name = "mistral-large-latest"

tokenizer = MistralTokenizer.from_model(model_name)

# Tokenize a list of messages
tokenized = tokenizer.encode_chat_completion(
    ChatCompletionRequest(
        tools=[
            Tool(
                function=Function(
                    name="get_current_weather",
                    description="Get the current weather",
                    parameters={
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "The city and state, e.g. San Francisco, CA",
                            },
                            "format": {
                                "type": "string",
                                "enum": ["celsius", "fahrenheit"],
                                "description": "The temperature unit to use. Infer this from the users location.",
                            },
                        },
                        "required": ["location", "format"],
                    },
                )
            )
        ],
        messages=[
            UserMessage(content="What's the weather like today in Paris"),
        ],
        model=model_name,
    )
)
tokens, text = tokenized.tokens, tokenized.text

# Count the number of tokens
print(len(tokens))

135


## הלמידה לא נעצרת כאן, המשיכו במסע

לאחר שסיימתם את השיעור הזה, בדקו את [אוסף הלמידה על AI גנרטיבי](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) כדי להמשיך להעשיר את הידע שלכם ב-AI גנרטיבי!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
